# Song Category Preprocessor

## What does this notebook do?

This notebook **prepares the test dataset** used by the Jingo multi-label classifier.  
It takes raw song CSV files (with lyrics), cleans the text, and labels each song with vocabulary categories (e.g. Animals, Food, Sports) based on keyword matching.

The output CSV is then used by the **training notebook** to evaluate the model.

---

### Pipeline

```
test.csv  ──┐
            ├──► Merge into one CSV ──► Clean lyrics ──► Assign categories ──► Save processed CSV
More_song_test.csv ──┘
```

### What each step does

| Step | What happens |
|---|---|
| **Merge** | Combines the base test set with extra test songs into one file |
| **Clean lyrics** | Lower-case, remove punctuation, protect multi-word keywords |
| **Assign categories** | Check each song's words against hand-crafted keyword lists |
| **Save** | Writes a new CSV with `categories` and `category_words` columns added |

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import re    # Regular expressions — used to strip punctuation and fix possessives
import json  # Converts Python lists ↔ JSON strings so they can be stored in CSV cells

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd  # DataFrames: lets us read, edit, and save CSV files easily

# ── Project-specific ──────────────────────────────────────────────────────────
from category import categories, multi_word_keywords
# categories         → a dict: { "Animals": ["cat","dog",...], "Food": [...], ... }
# multi_word_keywords → phrases like "hot dog" that must stay as one token during cleaning


# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit these paths/values to adjust the pipeline
# ═══════════════════════════════════════════════════════════════════════════════

# Input CSV files (raw, unprocessed)
TEST_BASE_FILE       = "../../data/test.csv"             # The main test set
TEST_ADDITIONAL_FILE = "../../data/More_song_test.csv"   # Extra test songs added later

# Output CSV files (written by this notebook)
TEST_MERGED_FILE     = "../../data/test_with_additional.csv"     # Merged test set (base + extra)
PROCESSED_TEST_OUTPUT = "../../data/processed_test_dataset.csv"  # Fully processed test set

# Column names
LYRICS_COLUMN = "Lyrics"  # Name of the column in the CSV that holds the raw lyrics text

# Matching threshold:
# A category is assigned only if at least this many of its keywords appear in the song.
# 1 means even a single matching keyword is enough to tag the song with that category.
MIN_MATCHED_WORDS = 1

Processed dataset saved to ../data/processed_test_dataset.csv


---
## Section 1 — Lyrics Cleaning & Category Assignment

### `clean_lyrics(lyrics)`
Takes a raw lyrics string and returns a **clean, normalised version** ready for keyword matching:

- Converts to **lower-case** — so "Cat" and "cat" are both recognised.
- Removes **possessives** — e.g. *cat's* → *cat* so the keyword `cat` still matches.
- Joins **multi-word keywords** with underscores — e.g. *hot dog* → *hot_dog* so TF-IDF treats it as a single token and keyword matching works correctly.
- Strips all **punctuation and symbols** — keeps only letters, digits, and spaces.

### `assign_categories_and_words(lyrics)`
Checks every unique word in the cleaned lyrics against every category's keyword list.  
If at least `MIN_MATCHED_WORDS` keywords from a category are found, that category is assigned to the song.

Returns two parallel lists:
- `assigned_categories` — e.g. `["Animals", "Nature"]`
- `category_words` — the specific keywords that triggered each category

In [ ]:
def clean_lyrics(lyrics):
    """
    Normalise a raw lyrics string so it is ready for keyword matching.

    Steps (applied in order):
      1. Force lower-case    →  "Cat" and "cat" are treated the same
      2. Remove possessives  →  "cat's" becomes "cat", "dogs'" becomes "dogs"
      3. Protect multi-word keywords by joining with underscores
         →  "hot dog" becomes "hot_dog" so it stays as one token
      4. Strip all non-alphanumeric characters (punctuation, symbols)

    Parameters
    ----------
    lyrics : any
        Raw value from the CSV. We call str() first in case it is NaN or a number.

    Returns
    -------
    str
        Clean lower-case lyrics with no punctuation.
    """
    lyrics = str(lyrics).lower()   # Step 1 — force lower-case, handle non-string values

    # Step 2a — remove possessive 's:  "cat's" → "cat"
    lyrics = re.sub(r"\b(\w+)'s\b", r"\1", lyrics)
    # Step 2b — remove trailing apostrophe:  "dogs'" → "dogs"
    lyrics = re.sub(r"\b(\w+)'\b", r"\1", lyrics)

    # Step 3 — protect multi-word keywords
    # Without this, "hot dog" would be two separate tokens and the keyword wouldn't match.
    for phrase in multi_word_keywords:
        lyrics = lyrics.replace(phrase, phrase.replace(" ", "_"))

    # Step 4 — remove every character that is NOT a letter, digit, underscore, or space
    lyrics = re.sub(r"[^\w\s]", "", lyrics)

    return lyrics


def assign_categories_and_words(lyrics):
    """
    Tag a song with vocabulary categories based on keyword presence in its lyrics.

    For every category defined in `categories` (loaded from category.py), we check
    whether any of that category's keywords appear in the lyrics.
    If the number of matches is >= MIN_MATCHED_WORDS, the song gets that category.

    Using a Python `set` for the words gives O(1) membership tests, so the
    intersection check is fast even for very long lyrics.

    Parameters
    ----------
    lyrics : str
        Already-cleaned lyrics string (output of `clean_lyrics`).

    Returns
    -------
    assigned_categories : list[str]
        Category names that matched.  Returns ["None"] if nothing matched.
    category_words : list[list[str]]
        For each matched category, the specific keywords found in the lyrics.
        Returns ["None"] if nothing matched.
    """
    words = set(lyrics.split())   # set → each unique word appears only once, fast lookup
    assigned_categories = []
    category_words = []

    for category, keywords in categories.items():
        # set intersection: gives us only the words that appear in BOTH the lyrics AND the keyword list
        matched_words = list(words & set(keywords))

        if len(matched_words) >= MIN_MATCHED_WORDS:   # threshold check (default: at least 1)
            assigned_categories.append(category)
            category_words.append(matched_words)

    if not assigned_categories:
        # No category matched → use sentinel values so the CSV column is never empty
        return ["None"], ["None"]

    return assigned_categories, category_words

---
## Section 2 — Dataset Pre-processing

### `preprocess_dataset(input_file, output_file)`
The **main processing function**. It:

1. Reads a raw songs CSV (needs `Song`, `Artist`, `Genre`, `Lyrics` columns).
2. Cleans every lyric string using `clean_lyrics`.
3. Assigns category labels and records which keywords matched using `assign_categories_and_words`.
4. Converts the `category_words` list to a JSON string so it can be stored in a CSV cell.
5. Drops unneeded columns and saves the result.

**Output columns:**

| Column | Description |
|---|---|
| `Song` | Song title |
| `Artist` | Artist name |
| `Genre` | Music genre |
| `categories` | List of matched category names, e.g. `['Animals', 'Food']` |
| `category_words` | JSON array of matched keywords per category |
| `cleaned_lyrics` | Normalised lyrics text used by the model |

In [ ]:
def preprocess_dataset(input_file, output_file, lyrics_column="Lyrics"):
    """
    Read a raw songs CSV, process every lyric, assign categories, and save the result.

    This is the core function of the notebook.  It turns a plain songs CSV into
    a labelled dataset that the classifier can evaluate against.

    Parameters
    ----------
    input_file    : str  Path to the input CSV (must have Song, Artist, Genre, Lyrics columns).
    output_file   : str  Path where the processed output CSV will be written.
    lyrics_column : str  Name of the column holding raw lyrics text (default "Lyrics").

    Returns
    -------
    pd.DataFrame | None
        The processed DataFrame, or None if an error occurred (error is printed).
    """
    try:
        # ── Step 1: Load the CSV ──────────────────────────────────────────────
        df = pd.read_csv(input_file)
        print(f"Loaded {len(df)} songs from {input_file}")

        # ── Step 2: Validate that the required columns exist ─────────────────
        # If any column is missing we print a helpful message and raise an error
        # rather than crashing with a confusing KeyError later.
        required_columns = [lyrics_column, "Song", "Artist", "Genre"]
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            print(f"Error: Missing columns {missing_columns} in {input_file}")
            print("Available columns:", df.columns.tolist())
            raise KeyError(f"Missing columns: {missing_columns}")

        # ── Step 3: Clean the lyrics ─────────────────────────────────────────
        # Applies lower-case, removes punctuation, protects multi-word keywords.
        # .apply() runs clean_lyrics on every row automatically.
        df["cleaned_lyrics"] = df[lyrics_column].apply(clean_lyrics)

        # ── Step 4: Assign category labels ───────────────────────────────────
        # assign_categories_and_words returns TWO values per row.
        # pd.Series unpacks them into two separate columns in one step.
        df[["categories", "category_words"]] = df["cleaned_lyrics"].apply(
            lambda x: pd.Series(assign_categories_and_words(x))
        )

        # ── Step 5: Serialise category_words to JSON strings ─────────────────
        # Python lists cannot be stored directly in CSV cells, so we convert them
        # to JSON strings.  Songs with no category get an empty JSON array "[]".
        df["category_words"] = df["category_words"].apply(
            lambda x: json.dumps(x) if x != ["None"] else "[]"
        )

        # ── Step 6: Keep only the necessary columns ───────────────────────────
        output_columns = ["Song", "Artist", "Genre", "categories", "category_words", "cleaned_lyrics"]
        df = df[output_columns]

        # ── Step 7: Save to disk ──────────────────────────────────────────────
        df.to_csv(output_file, index=False)
        print(f"Processed dataset saved to {output_file}  ({len(df)} songs)")
        return df

    except FileNotFoundError:
        # Friendly message when the input file path is wrong
        print(f"Error: The file '{input_file}' was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        return None

---
## Section 3 — Run the Pre-processing

**Run this cell to execute the full pipeline.**

What it does:
1. Loads `test.csv` (base test set) and `More_song_test.csv` (extra songs).
2. Merges them into one combined CSV with `pd.concat`.
3. Calls `preprocess_dataset` which cleans lyrics, assigns categories, and saves the final output.

> The output file (`processed_test_dataset.csv`) is what the training notebook loads when evaluating the classifier.

In [ ]:
# ── Step 1: Load the two test CSV files ──────────────────────────────────────
original_test = pd.read_csv(TEST_BASE_FILE)            # The main test set
additional    = pd.read_csv(TEST_ADDITIONAL_FILE)      # Extra test songs

print(f"Base test songs      : {len(original_test)}")
print(f"Additional test songs: {len(additional)}")

# ── Step 2: Merge into one combined dataset ───────────────────────────────────
# pd.concat stacks the two DataFrames vertically (one after the other).
# ignore_index=True re-numbers the rows from 0 so there are no duplicate indices.
merged = pd.concat([original_test, additional], ignore_index=True)
print(f"Total after merge    : {len(merged)} songs")

# Save the merged file so it can be inspected or re-used without re-merging
merged.to_csv(TEST_MERGED_FILE, index=False)
print(f"Merged test set saved to {TEST_MERGED_FILE}")

# ── Step 3: Pre-process the merged dataset ────────────────────────────────────
# This cleans the lyrics, assigns category labels, and saves the processed CSV.
preprocess_dataset(
    input_file    = TEST_MERGED_FILE,      # the merged file we just created
    output_file   = PROCESSED_TEST_OUTPUT, # where to write the labelled output
    lyrics_column = LYRICS_COLUMN,         # column that holds the raw lyrics text
)